# Export RandomForest Model for Live Trading

Loads `data/latest_features.jsonl`, trains RF with 20 optimal features (from notebook 11),
exports model + scaler + feature columns to `models/`.

In [1]:
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

random.seed(42)
np.random.seed(42)

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)
FEATURES_PATH = Path("../data/latest_features.jsonl")

## 1. Load features

In [2]:
rows = []
with open(FEATURES_PATH) as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)

# 20 optimal features (from notebook 11 RF importance ranking)
feat_cols = [
    "down_risk_reward",
    "up_risk_reward",
    "btc_move_from_open",
    "down_implied_probability",
    "rr_spread",
    "up_implied_probability",
    "down_token_velocity",
    "up_spread_level",
    "up_token_velocity",
    "btc_velocity",
    "adx",
    "time_of_day_cos",
    "volume_momentum",
    "ma_crossover",
    "regime_score",
    "stochastic_k",
    "return_autocorrelation",
    "conviction_score",
    "multi_candle_return_6",
    "rsi",
]
df[feat_cols] = df[feat_cols].fillna(0.0)

print(f"Loaded {len(df):,} rows, {df['candle_id'].nunique()} candles")
print(f"Using {len(feat_cols)} optimal features")
print(f"UP rate: {df['target'].mean() * 100:.1f}%")

Loaded 53,377 rows, 1103 candles
Using 20 optimal features
UP rate: 51.8%


## 2. Train/val split and evaluate

In [3]:
candle_ids = df["candle_id"].unique()
split_idx = int(len(candle_ids) * 0.8)
train_ids = set(candle_ids[:split_idx])

df_train = df[df["candle_id"].isin(train_ids)]
df_val = df[~df["candle_id"].isin(train_ids)]

scaler = StandardScaler()
X_train = scaler.fit_transform(df_train[feat_cols].values)
y_train = df_train["target"].values
X_val = scaler.transform(df_val[feat_cols].values)
y_val = df_val["target"].values

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train))
val_acc = accuracy_score(y_val, model.predict(X_val))

print(f"Train accuracy: {train_acc * 100:.1f}%")
print(f"Val accuracy:   {val_acc * 100:.1f}%")
print("\nValidation report:")
print(classification_report(y_val, model.predict(X_val), target_names=["DOWN", "UP"]))

# Per-candle accuracy
probs = model.predict_proba(X_val)[:, 1]
correct, total = 0, 0
for cid in df_val["candle_id"].unique():
    cmask = df_val["candle_id"] == cid
    y_c = df_val.loc[cmask, "target"].values[0]
    vote = 1 if probs[cmask.values].mean() >= 0.5 else 0
    total += 1
    if vote == y_c:
        correct += 1
print(f"Per-candle accuracy: {correct}/{total} = {correct / total * 100:.1f}%")

Train accuracy: 94.8%
Val accuracy:   73.3%

Validation report:
              precision    recall  f1-score   support

        DOWN       0.79      0.68      0.73      5541
          UP       0.69      0.79      0.74      4911

    accuracy                           0.73     10452
   macro avg       0.74      0.74      0.73     10452
weighted avg       0.74      0.73      0.73     10452

Per-candle accuracy: 195/221 = 88.2%


## 3. Export (retrained on all data)

In [4]:
X_all = scaler.fit_transform(df[feat_cols].values)
y_all = df["target"].values
model.fit(X_all, y_all)

joblib.dump(model, MODELS_DIR / "rf_v1.joblib")
joblib.dump(scaler, MODELS_DIR / "rf_scaler_v1.joblib")
joblib.dump(feat_cols, MODELS_DIR / "rf_feature_cols_v1.joblib")

print(f"Saved to {MODELS_DIR} (retrained on all {len(df):,} rows):")
for f in sorted(MODELS_DIR.glob("rf_*_v1.*")):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")

Saved to ../models (retrained on all 53,377 rows):
  rf_feature_cols_v1.joblib (373 bytes)
  rf_scaler_v1.joblib (1,063 bytes)


## 4. Verify load

In [5]:
m2 = joblib.load(MODELS_DIR / "rf_v1.joblib")
s2 = joblib.load(MODELS_DIR / "rf_scaler_v1.joblib")
fc2 = joblib.load(MODELS_DIR / "rf_feature_cols_v1.joblib")

sample = df[fc2].iloc[:1].fillna(0.0).values
prob_orig = model.predict_proba(scaler.transform(sample))[0, 1]
prob_loaded = m2.predict_proba(s2.transform(sample))[0, 1]

print(f"Original prob:  {prob_orig:.6f}")
print(f"Loaded prob:    {prob_loaded:.6f}")
print(f"Match: {prob_orig == prob_loaded}")
print(f"\nFeature columns ({len(fc2)}): {fc2}")

Original prob:  0.700518
Loaded prob:    0.700518
Match: True

Feature columns (20): ['down_risk_reward', 'up_risk_reward', 'btc_move_from_open', 'down_implied_probability', 'rr_spread', 'up_implied_probability', 'down_token_velocity', 'up_spread_level', 'up_token_velocity', 'btc_velocity', 'adx', 'time_of_day_cos', 'volume_momentum', 'ma_crossover', 'regime_score', 'stochastic_k', 'return_autocorrelation', 'conviction_score', 'multi_candle_return_6', 'rsi']
